In [3]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
print(ROOT)

c:\Users\HP\Desktop\Ejust\Projects\Data Security\nids


In [5]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.features import to_matrix
from src.pipeline import DATA, RESULTS, load_scaler, iter_split

SEED = 11
cols, scaler = load_scaler()
print(ROOT, len(cols), "features")

c:\Users\HP\Desktop\Ejust\Projects\Data Security\nids 40 features


In [7]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.thresholds import percentile_thr, mean_k_thr, mad_thr
from src.evaluation import tpr_at_threshold

SC = ROOT / "results" / "scores"
files = {
    "M1 IsolationForest": "m1_iforest_nftoniot_seed11.parquet",
    "M2 OneClassSVM": "m2_ocsvm_nftoniot_seed11.parquet",
    "M3 AE v1": "m3_autoencoder_nftoniot_seed11.parquet",
    "M3 AE v2": "m3v2_autoencoder_nftoniot_seed11.parquet",
}
methods = {
    "percentile 99": lambda s: percentile_thr(s, 0.99),
    "mean + 3 std": lambda s: mean_k_thr(s, 3.0),
    "MAD (k=3.5)": lambda s: mad_thr(s, 3.5),
}

rows = []
for name, f in files.items():
    df = pd.read_parquet(SC / f)
    val = df[(df["split"] == "val") & (df["label"] == 0)]["score"].to_numpy()
    for mname, fn in methods.items():
        thr = fn(val)                                   # always set on benign val only
        for sp in ["tune", "test"]:
            d = df[df["split"] == sp]
            if d.empty:
                continue
            tpr, fpr = tpr_at_threshold(d, thr)
            macro = np.mean([(g["score"] > thr).mean()
                             for _, g in d[d["label"] == 1].groupby("attack_type")])
            rows.append({"model": name, "threshold": mname, "split": sp,
                         "fpr": fpr, "tpr": tpr, "macro_tpr": macro})

table = pd.DataFrame(rows).round(3)
table.to_csv(ROOT / "results" / "threshold_comparison.csv", index=False)
table[table["split"] == "tune"]

,model,threshold,split,fpr,tpr,macro_tpr
0,M1 IsolationForest,percentile 99,tune,0.011,0.005,0.020
2,M1 IsolationForest,mean + 3 std,tune,0.020,0.016,0.085
4,M1 IsolationForest,MAD (k=3.5),tune,0.166,0.125,0.868
6,M2 OneClassSVM,percentile 99,tune,0.015,0.004,0.024
8,M2 OneClassSVM,mean + 3 std,tune,0.038,0.036,0.475
10,M2 OneClassSVM,MAD (k=3.5),tune,0.106,0.594,0.897
12,M3 AE v1,percentile 99,tune,0.013,0.018,0.058
14,M3 AE v1,mean + 3 std,tune,0.001,0.001,0.006
16,M3 AE v1,MAD (k=3.5),tune,0.238,1.000,1.000
18,M3 AE v2,percentile 99,tune,0.013,0.007,0.063


In [8]:
table[table["split"] == "test"]

,model,threshold,split,fpr,tpr,macro_tpr
1,M1 IsolationForest,percentile 99,test,0.073,0.100,0.085
3,M1 IsolationForest,mean + 3 std,test,0.156,0.244,0.193
5,M1 IsolationForest,MAD (k=3.5),test,0.946,0.889,0.922
7,M2 OneClassSVM,percentile 99,test,0.422,0.281,0.321
9,M2 OneClassSVM,mean + 3 std,test,0.638,0.541,0.579
11,M2 OneClassSVM,MAD (k=3.5),test,0.849,0.868,0.896
13,M3 AE v1,percentile 99,test,0.269,0.236,0.370
15,M3 AE v1,mean + 3 std,test,0.108,0.075,0.078
17,M3 AE v1,MAD (k=3.5),test,0.955,0.999,0.999
19,M3 AE v2,percentile 99,test,0.536,0.242,0.378


In [ ]:
from src.pipeline import iter_split

def get(name, cap=30_000, n_benign=100_000):
    df = pd.concat(list(iter_split(name, cols, cap=cap)))
    b = df[df["Attack"] == "Benign"]
    b = b.sample(min(n_benign, len(b)), random_state=SEED)
    return pd.concat([b, df[df["Attack"] != "Benign"]])

tr = pd.read_parquet(DATA / "train.parquet", columns=cols + ["Attack"]).sample(200_000, random_state=SEED)
va, tu, te = get("val"), get("tune"), get("test")

groups = {
    "train_benign": tr,
    "val_benign": va,
    "tune_benign": tu[tu["Attack"] == "Benign"],
    "tune_attack": tu[tu["Attack"] != "Benign"],
    "test_benign": te[te["Attack"] == "Benign"],
    "test_attack": te[te["Attack"] != "Benign"],
}
feats = [c for c in ["DURATION_OUT", "DURATION_IN", "RETRANSMITTED_IN_PKTS",
                     "RETRANSMITTED_IN_BYTES", "IN_BYTES", "LONGEST_FLOW_PKT"] if c in cols]

print("fraction of flows with value > 0")
print(pd.DataFrame({g: (d[feats] > 0).mean() for g, d in groups.items()}).round(3))
print("\n99th percentile (raw values)")
print(pd.DataFrame({g: d[feats].quantile(0.99) for g, d in groups.items()}).round(1))
print("\ntrain std after log1p (what the scaler divides by)")
print(pd.Series(scaler.scale_, index=cols)[feats].round(4))

fraction of flows with value > 0
                        train_benign  val_benign  tune_benign  tune_attack  \
DURATION_OUT                   0.002       0.003        0.004        0.010   
DURATION_IN                    0.008       0.017        0.011        0.018   
RETRANSMITTED_IN_PKTS          0.000       0.000        0.001        0.001   
RETRANSMITTED_IN_BYTES         0.000       0.000        0.001        0.001   
IN_BYTES                       1.000       1.000        1.000        1.000   
LONGEST_FLOW_PKT               1.000       1.000        1.000        1.000   

                        test_benign  test_attack  
DURATION_OUT                  0.265        0.223  
DURATION_IN                   0.334        0.223  
RETRANSMITTED_IN_PKTS         0.047        0.026  
RETRANSMITTED_IN_BYTES        0.047        0.026  
IN_BYTES                      1.000        1.000  
LONGEST_FLOW_PKT              1.000        1.000  

99th percentile (raw values)
                        train_ben